# Robustness Summary Tables

Reads `results/robustness/sensitivity_summary.csv` and produces the three tables that appear in the Supplementary Material:

- **Table S1** — Effective-rank difference by regularization floor $\epsilon$ (c=0.01, infinite horizon, k∈{1,2,3,5})
- **Table S2** — Effective-rank difference by finite horizon and stabilization margin
- **Table S3** — Effective-rank difference by driver-set size k

Also reports the matched-proportion (5%) landscape near-optimal fractions from `results/results_fraction_5pct/robustness/sensitivity_summary.csv`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

REPO = Path("..")
df = pd.read_csv(REPO / "results/robustness/sensitivity_summary.csv")
df5 = pd.read_csv(REPO / "results/results_fraction_5pct/robustness/sensitivity_summary.csv")

SCALES = [68, 114, 219]
PRIMARY_K = [1, 2, 3, 5]

## Helper

In [2]:
def geometry_diff(sub, metric="effective_rank"):
    """Return per-subject mean difference (cycle minus degree) for a filtered slice."""
    cy = sub[sub.method == "cycle_participation"].groupby("subject_id")[metric].mean()
    de = sub[sub.method == "degree_strength"].groupby("subject_id")[metric].mean()
    return cy - de


def summarise(diff):
    """Return (mean, n_positive, p_value) for a series of paired differences."""
    diff = diff.dropna()
    if len(diff) < 2:
        return np.nan, np.nan, np.nan
    _, p = stats.wilcoxon(diff)
    return diff.mean(), (diff > 0).sum(), p

---
## Matched-proportion landscape (≈5% of network)

Driver-set sizes: k=3 (68 regions), k=6 (114 regions), k=11 (219 regions). Near-optimal threshold: within 5% of best observed energy.

In [3]:
scale_k = {68: 3, 114: 6, 219: 11}
rows = []
for scale, k in scale_k.items():
    sub = df5[(df5.scale == scale) & (df5.k == k) & (df5.c_value == 0.1)]
    rows.append({
        "Scale (regions)": scale,
        "k (≈5%)": k,
        "Mean near-optimal fraction (5% threshold)": f"{sub.near_optimal_fraction_05pct.mean():.4f}",
        "SEM": f"{sub.near_optimal_fraction_05pct.sem():.4f}",
        "n subjects": len(sub),
    })

pd.DataFrame(rows).set_index("Scale (regions)")

,k (≈5%),Mean near-optimal fraction (5% threshold),SEM,n subjects
Scale (regions),,,,
68,3,0.9617,0.0026,140
114,6,0.9974,0.0004,140
219,11,1.0000,0.0000,140


---
## Table S1 — Epsilon sensitivity

Effective-rank difference (cycle minus degree), mean across subjects, averaged over k∈{1,2,3,5}.  
Condition: infinite horizon, c=0.01.

In [4]:
stab = df[
    (df.sensitivity_family == "stabilization_epsilon")
    & (df.c_value == 0.01)
    & (df.k.isin(PRIMARY_K))
]

eps_values = sorted(stab.epsilon.dropna().unique())
rows = []
for eps in eps_values:
    row = {"epsilon": f"{eps:.0e}"}
    for scale in SCALES:
        sub = stab[(stab.epsilon == eps) & (stab.scale == scale)]
        diff = geometry_diff(sub)
        mean, n_pos, p = summarise(diff)
        row[f"{scale} regions"] = f"{mean:.3f}"
        row[f"{scale} n_pos"] = f"{int(n_pos)}/70"
        row[f"{scale} p"] = f"{p:.1e}"
    rows.append(row)

table_s1 = pd.DataFrame(rows).set_index("epsilon")
# Compact view: mean only
mean_cols = [c for c in table_s1.columns if "regions" in c and "n_pos" not in c and " p" not in c]
print("Mean effective-rank difference (cycle minus degree):")
table_s1[mean_cols]

Mean effective-rank difference (cycle minus degree):


,68 regions,114 regions,219 regions
epsilon,,,
1e-08,1.053,1.294,1.669
1e-07,1.053,1.294,1.670
1e-06,1.054,1.296,1.676
1e-05,1.061,1.317,1.732
1e-04,1.122,1.486,2.225
1e-03,1.663,3.288,9.265
1e-02,7.700,24.110,82.463


In [5]:
# Full table with n_positive and p-value
table_s1

,68 regions,68 n_pos,68 p,114 regions,114 n_pos,114 p,219 regions,219 n_pos,219 p
epsilon,,,,,,,,,
1e-08,1.053,69/70,8.8e-13,1.294,70/70,3.6e-13,1.669,70/70,3.6e-13
1e-07,1.053,69/70,8.8e-13,1.294,70/70,3.6e-13,1.670,70/70,3.6e-13
1e-06,1.054,69/70,8.8e-13,1.296,70/70,3.6e-13,1.676,70/70,3.6e-13
1e-05,1.061,69/70,8.4e-13,1.317,70/70,3.6e-13,1.732,70/70,3.6e-13
1e-04,1.122,69/70,8.4e-13,1.486,70/70,3.6e-13,2.225,70/70,3.6e-13
1e-03,1.663,69/70,7.4e-13,3.288,70/70,3.6e-13,9.265,70/70,3.6e-13
1e-02,7.700,69/70,6.8e-13,24.110,70/70,3.6e-13,82.463,70/70,3.6e-13


---
## Table S2 — Finite horizon sensitivity

Effective-rank difference (cycle minus degree), mean across subjects, averaged over k∈{1,2,3,5}.  
Both c values; epsilon=1e-8 (smallest floor, most conservative).

In [6]:
finite = df[
    (df.sensitivity_family == "finite_horizon")
    & (df.epsilon_case == "epsilon_floor")
    & (df.k.isin(PRIMARY_K))
]

# epsilon values available in finite_horizon+floor
print("Available epsilon values in finite_horizon+epsilon_floor:")
print(sorted(finite.epsilon.dropna().unique()))

Available epsilon values in finite_horizon+epsilon_floor:
[np.float64(1e-08), np.float64(0.01)]


In [7]:
# Use minimum epsilon (most conservative)
min_eps = finite.epsilon.dropna().min()
finite_sub = finite[finite.epsilon == min_eps]

rows = []
for c in [0.01, 1.0]:
    for horizon in [1.0, 2.5, 5.0]:
        row = {"c": c, "Horizon T": horizon}
        for scale in SCALES:
            sub = finite_sub[
                (finite_sub.c_value == c)
                & (finite_sub.horizon == horizon)
                & (finite_sub.scale == scale)
            ]
            diff = geometry_diff(sub)
            mean, n_pos, p = summarise(diff)
            row[f"{scale} regions"] = f"{mean:.2f}"
            row[f"{scale} n_pos"] = f"{int(n_pos)}/70"
            row[f"{scale} p"] = f"{p:.1e}"
        rows.append(row)

table_s2 = pd.DataFrame(rows).set_index(["c", "Horizon T"])
mean_cols = [c for c in table_s2.columns if "regions" in c and "n_pos" not in c and " p" not in c]
print("Mean effective-rank difference (cycle minus degree):")
table_s2[mean_cols]

Mean effective-rank difference (cycle minus degree):


68 regions 114 regions 219 regions
c    Horizon T                                   
0.01 1.0             0.07        0.06       -0.16
     2.5             0.37        0.44        0.12
     5.0             0.73        0.87        0.63
1.00 1.0            -0.02       -0.05       -0.20
     2.5            -0.02       -0.06       -0.25
     5.0            -0.02       -0.06       -0.25

In [8]:
# Full table
table_s2

68 regions 68 n_pos     68 p 114 regions 114 n_pos    114 p  \
c    Horizon T                                                               
0.01 1.0             0.07    40/70  3.6e-03        0.06     44/70  9.1e-03   
     2.5             0.37    66/70  1.8e-12        0.44     66/70  1.9e-12   
     5.0             0.73    70/70  3.6e-13        0.87     70/70  3.6e-13   
1.00 1.0            -0.02    33/70  2.4e-01       -0.05     23/70  1.4e-03   
     2.5            -0.02    34/70  3.1e-01       -0.06     24/70  2.8e-03   
     5.0            -0.02    34/70  3.1e-01       -0.06     24/70  2.7e-03   

               219 regions 219 n_pos    219 p  
c    Horizon T                                 
0.01 1.0             -0.16     10/70  1.1e-09  
     2.5              0.12     44/70  2.9e-03  
     5.0              0.63     63/70  2.4e-12  
1.00 1.0             -0.20      4/70  8.1e-13  
     2.5             -0.25      3/70  7.7e-13  
     5.0             -0.25      3/70  7.7e-13

### Infinite-horizon result at c=1.0 (the reversal case)

For completeness, also show the c=1.0 infinite-horizon result across epsilon values.

In [9]:
stab_c1 = df[
    (df.sensitivity_family == "stabilization_epsilon")
    & (df.c_value == 1.0)
    & (df.k.isin(PRIMARY_K))
]

rows = []
for eps in eps_values:
    row = {"epsilon": f"{eps:.0e}"}
    for scale in SCALES:
        sub = stab_c1[(stab_c1.epsilon == eps) & (stab_c1.scale == scale)]
        diff = geometry_diff(sub)
        mean, n_pos, p = summarise(diff)
        row[f"{scale} regions"] = f"{mean:.3f}"
        row[f"{scale} n_pos"] = f"{int(n_pos)}/70"
        row[f"{scale} p"] = f"{p:.1e}"
    rows.append(row)

table_c1 = pd.DataFrame(rows).set_index("epsilon")
mean_cols = [c for c in table_c1.columns if "regions" in c and "n_pos" not in c and " p" not in c]
print("c=1.0, infinite horizon — mean effective-rank difference (cycle minus degree):")
table_c1[mean_cols]

c=1.0, infinite horizon — mean effective-rank difference (cycle minus degree):


,68 regions,114 regions,219 regions
epsilon,,,
1e-08,-0.019,-0.059,-0.249
1e-07,-0.019,-0.059,-0.249
1e-06,-0.019,-0.058,-0.248
1e-05,-0.016,-0.052,-0.237
1e-04,0.010,0.009,-0.099
1e-03,0.340,1.055,3.808
1e-02,2.890,5.948,10.436


---
## Table S3 — Larger driver-set sizes

Effective-rank difference (cycle minus degree) per subject at each k.  
Condition: c=0.01, epsilon=1e-5, infinite horizon.

In [10]:
large_k = df[
    (df.sensitivity_family == "stabilization_epsilon")
    & (df.c_value == 0.01)
    & (df.epsilon == 1e-5)
]

rows = []
for scale in SCALES:
    sc = large_k[large_k.scale == scale]
    for k in sorted(sc.k.unique()):
        sub = sc[sc.k == k]
        diff = geometry_diff(sub)
        mean, n_pos, p = summarise(diff)
        rows.append({
            "Scale (regions)": scale,
            "k": k,
            "Mean eff-rank diff": round(mean, 3),
            "n_positive": f"{int(n_pos)}/70",
            "p (Wilcoxon)": f"{p:.2e}",
        })

table_s3 = pd.DataFrame(rows)
table_s3

,Scale (regions),k,Mean eff-rank diff,n_positive,p (Wilcoxon)
0,68,1,0.454,60/70,7.08e-11
1,68,2,0.900,64/70,3.41e-12
2,68,3,1.299,69/70,1.04e-12
3,68,5,1.592,69/70,3.88e-13
4,68,7,1.966,68/70,4.05e-13
5,68,10,2.192,70/70,3.56e-13
6,68,14,2.139,69/70,3.72e-13
7,114,1,0.395,60/70,5.43e-11
8,114,2,1.085,67/70,9.62e-13
9,114,3,1.626,69/70,3.72e-13


In [11]:
# Pivot to wide format matching the supplement table layout
pivot = table_s3.pivot(index="k", columns="Scale (regions)", values="Mean eff-rank diff")
pivot.columns = [f"{c} regions" for c in pivot.columns]
print("Mean effective-rank difference (cycle minus degree) by k:")
pivot

Mean effective-rank difference (cycle minus degree) by k:


,68 regions,114 regions,219 regions
k,,,
1,0.454,0.395,0.417
2,0.900,1.085,1.421
3,1.299,1.626,2.007
5,1.592,2.161,3.082
6,NaN,2.403,NaN
7,1.966,NaN,NaN
10,2.192,3.048,5.235
11,NaN,2.984,5.642
14,2.139,NaN,NaN
